In [ ]:
!pip install faker

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
import random
import os
import json
from datetime import datetime, timedelta

In [ ]:
fake = Faker()
Faker.seed(42)
np.random.seed(42)

# Configuration
NUM_ADVISORS = 50
NUM_CLIENTS = 5000
NUM_FUNDS = 20
NUM_MONTHS = 24
OUTPUT_DIR = "mock_data"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Generating Dimensions...")

# 1. Generate Advisors
advisors = []
for i in range(NUM_ADVISORS):
    client_load = random.choices(['Low', 'Medium', 'High'], weights=[0.2, 0.5, 0.3])[0]
    advisors.append({
        'Advisor_ID': f'ADV_{i+1:03}',
        'Advisor_Name': fake.name(),
        'Branch_ID': f'BR_{random.randint(1, 5):02}',
        'Hire_Date': fake.date_between(start_date='-10y', end_date='-2y'),
        'Capacity_Status': client_load
    })
df_advisors = pd.DataFrame(advisors)
df_advisors.to_csv(f"{OUTPUT_DIR}/Dim_Advisor.csv", index=False)

# 2. Generate Clients
clients = []
for i in range(NUM_CLIENTS):
    # Skew client distribution: 'High' capacity advisors get way more clients
    adv = df_advisors.sample(1, weights=df_advisors['Capacity_Status'].map({'Low': 1, 'Medium': 3, 'High': 8})).iloc[0]
    
    # Inject intentional NULLs for realism (5% missing retirement year)
    target_retirement = fake.date_between(start_date='+5y', end_date='+30y').year
    if random.random() < 0.05:
        target_retirement = np.nan
        
    clients.append({
        'Client_ID': f'CLI_{i+1:05}',
        'Advisor_ID': adv['Advisor_ID'],
        'Client_Name': fake.name(),
        'Risk_Tolerance': random.choice(['Conservative', 'Moderate', 'Aggressive']),
        'Target_Retirement_Year': target_retirement,
        'Churn_Risk_Score': round(random.uniform(0.1, 0.9), 2) if adv['Capacity_Status'] == 'High' else round(random.uniform(0.01, 0.4), 2)
    })
df_clients = pd.DataFrame(clients)
df_clients.to_csv(f"{OUTPUT_DIR}/Dim_Client.csv", index=False)

# 3. Generate Funds
funds = []
asset_classes = ['Equity', 'Fixed Income', 'Alternative', 'Cash']
for i in range(NUM_FUNDS):
    funds.append({
        'Fund_ID': f'FND_{i+1:03}',
        'Fund_Name': fake.company() + ' Fund',
        'Ticker': fake.unique.lexify(text='????').upper(),
        'Asset_Class': random.choice(asset_classes),
        'Expense_Ratio': round(random.uniform(0.05, 1.50), 2)
    })
df_funds = pd.DataFrame(funds)
df_funds.to_csv(f"{OUTPUT_DIR}/Dim_Fund.csv", index=False)

print("Generating Fact Data (This simulates the stagnation skew)...")

# 4. Generate AUM Fact Table
# Story: First 12 months show growth, last 12 months stagnate/decline, especially for High capacity advisors.
aum_records = []
start_date = datetime.now() - timedelta(days=NUM_MONTHS*30)

for month in range(NUM_MONTHS):
    current_date = start_date + timedelta(days=month*30)
    is_stagnation_period = month >= 12
    
    for _, client in df_clients.iterrows():
        # Only process a sample of clients each month to control volume
        if random.random() > 0.3: continue
            
        adv_capacity = df_advisors[df_advisors['Advisor_ID'] == client['Advisor_ID']]['Capacity_Status'].values[0]
        fund = random.choice(funds)
        
        # Base flow logic
        base_flow = random.uniform(-5000, 15000)
        
        # Apply the stagnation story
        if is_stagnation_period:
            if adv_capacity == 'High':
                base_flow -= random.uniform(5000, 25000) # Heavy outflows for overloaded advisors
            else:
                base_flow -= random.uniform(0, 5000) # Mild stagnation for others
                
        aum_records.append({
            'AUM_ID': fake.uuid4(),
            'Client_ID': client['Client_ID'],
            'Fund_ID': fund['Fund_ID'],
            'Date_ID': current_date.strftime('%Y-%m-%d'),
            'EOM_Total_Value': round(max(0, random.uniform(10000, 500000) + base_flow), 2),
            'Net_Flow': round(base_flow, 2)
        })

df_aum = pd.DataFrame(aum_records)
df_aum.to_csv(f"{OUTPUT_DIR}/Fact_AUM_Monthly.csv", index=False)

print("Generating Unstructured CRM Notes for Cortex Search...")

# 5. Generate Unstructured CRM Notes
crm_notes = []
for _, client in df_clients.sample(100).iterrows(): # Small subset to save Cortex AI costs
    crm_notes.append({
        'Note_ID': fake.uuid4(),
        'Client_ID': client['Client_ID'],
        'Advisor_ID': client['Advisor_ID'],
        'Interaction_Date': fake.date_between(start_date='-1y', end_date='today').strftime('%Y-%m-%d'),
        'Note_Text': f"Met with {client['Client_Name']} today. Discussed their target retirement in {client['Target_Retirement_Year']}. They are concerned about market volatility and mentioned wanting to reduce risk. Also noted a recent job change which might impact their 401k rollover."
    })
    
with open(f"{OUTPUT_DIR}/Doc_CRM_Notes.json", 'w') as f:
    json.dump(crm_notes, f, indent=4)

print(f"Data generation complete! Files saved to ./{OUTPUT_DIR}/")